# 1.3 softmax 及其梯度 / Softmax and its gradient

推出 p − y 这个结果，并写一个不会溢出的 softmax。

Derive the p − y result and write a softmax that does not overflow.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 数值稳定的 softmax 与 log-sum-exp / A numerically stable softmax and log-sum-exp

只用 `max`、`exp`、`log`、`sum` 实现 `logsumexp`、`log_softmax` 和 `softmax`。测试在运行你的代码期间会禁用 PyTorch 的内置版本，并用 logit 为 1000、10000 和 -10000 的输入检查是否溢出。

Implement `logsumexp`, `log_softmax` and `softmax` using only `max`, `exp`, `log` and `sum`. The tests disable the PyTorch built-ins while your code runs and check for overflow with logits of 1000, 10000 and -10000.

In [ ]:
import torch

# Do not call torch.logsumexp, torch.softmax, torch.log_softmax or their
# F.* / tensor-method versions. The tests disable them while your code runs.


def logsumexp(z, dim=-1, keepdim=False):
    """log(sum(exp(z))) along `dim`, without overflow."""
    # TODO 1: subtract the maximum before exponentiating, add it back afterwards
    raise NotImplementedError


def log_softmax(z, dim=-1):
    # TODO 2: use logsumexp
    raise NotImplementedError


def softmax(z, dim=-1):
    # TODO 3: use log_softmax
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
z = torch.randn(4, 7, dtype=torch.float64) * 5
cases = {
    "random": z,
    "huge": torch.tensor([[1000.0, 1000.0, 1000.0], [1e4, 0.0, -1e4]], dtype=torch.float64),
    "very negative": torch.tensor([[-1e4, -1e4], [-800.0, -805.0]], dtype=torch.float64),
}
# Compute the references first, then switch the built-ins off.
refs = {k: (torch.logsumexp(v, -1), torch.log_softmax(v, -1), torch.softmax(v, -1)) for k, v in cases.items()}
ref_dim0 = torch.log_softmax(z, 0)

def _banned(*a, **k):
    raise AssertionError("built-in softmax / logsumexp is not allowed in this exercise")

_saved = [(m, n, getattr(m, n)) for m in (torch, F, torch.Tensor) for n in ("logsumexp", "softmax", "log_softmax") if hasattr(m, n)]
for m, n, _ in _saved:
    setattr(m, n, _banned)
try:
    for name, v in cases.items():
        lse, ls, sm = refs[name]
        assert torch.allclose(logsumexp(v), lse, atol=1e-10), f"logsumexp wrong on '{name}' (nan or inf means overflow)"
        assert torch.allclose(log_softmax(v), ls, atol=1e-10), f"log_softmax wrong on '{name}'"
        assert torch.allclose(softmax(v), sm, atol=1e-10), f"softmax wrong on '{name}'"
    assert logsumexp(z, keepdim=True).shape == (4, 1)
    assert torch.allclose(log_softmax(z, dim=0), ref_dim0, atol=1e-10), "the dim argument is ignored"
    p = softmax(z)
    assert torch.allclose(p.sum(-1), torch.ones(4, dtype=torch.float64))
    # Adding a constant to every logit changes nothing.
    assert torch.allclose(softmax(z + 123.0), p, atol=1e-10)
    # Two equal logits share the mass, even at -10000.
    assert abs(log_softmax(cases["very negative"])[0, 0].item() + math.log(2)) < 1e-10
finally:
    for m, n, f in _saved:
        setattr(m, n, f)
print("m13_stable_softmax: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

先用 `z.max(dim=dim, keepdim=True).values` 取出最大值 `m`，对 `z - m` 取指数，最后把 `m` 加回来。

Take the maximum `m` with `z.max(dim=dim, keepdim=True).values`, exponentiate `z - m`, and add `m` back at the end.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`softmax` 不要自己重新写一遍除法，直接对 `log_softmax` 的结果取指数即可。

Do not write the division again for `softmax`. Exponentiate the result of `log_softmax`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

注意 `keepdim`。内部计算保持维度方便广播，返回前按参数决定是否 `squeeze`。

Mind `keepdim`. Keep the dimension internally so broadcasting works and squeeze before returning if asked.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def logsumexp(z, dim=-1, keepdim=False):
    m = z.max(dim=dim, keepdim=True).values           # largest entry, so exp(z - m) <= 1
    out = m + torch.log(torch.exp(z - m).sum(dim=dim, keepdim=True))
    return out if keepdim else out.squeeze(dim)


def log_softmax(z, dim=-1):
    return z - logsumexp(z, dim=dim, keepdim=True)


def softmax(z, dim=-1):
    return torch.exp(log_softmax(z, dim=dim))
```

</details>

## 2. 手写 softmax 交叉熵的反向传播 / Backpropagation through softmax cross-entropy by hand

不使用 autograd，写出 softmax 的雅可比矩阵、平均交叉熵对 logits 的梯度，以及一层线性层 `logits = x @ W.T + b` 的全部梯度。测试在 `torch.no_grad()` 下调用你的函数，并与 autograd 的结果对比。

Without autograd, write the softmax Jacobian, the gradient of the mean cross-entropy with respect to the logits, and all gradients of a linear layer `logits = x @ W.T + b`. The tests call your functions under `torch.no_grad()` and compare with autograd.

In [ ]:
import torch

# No autograd in this exercise. The tests call your functions inside
# torch.no_grad(), so .backward() will not work. Write the formulas.


def softmax_jacobian(z):
    """z: (V,). Returns J of shape (V, V) with J[i, j] = d p_i / d z_j."""
    # TODO 1: J = diag(p) - p p^T
    raise NotImplementedError


def ce_grad_logits(logits, targets):
    """Gradient of the MEAN cross-entropy with respect to the logits.

    logits: (N, V), targets: (N,) long. Returns (N, V).
    """
    # TODO 2: (p - onehot(targets)) / N
    raise NotImplementedError


def linear_ce_backward(x, W, b, targets):
    """One linear layer followed by softmax cross-entropy.

    logits = x @ W.T + b     with x: (N, D), W: (V, D), b: (V,)
    Returns (loss, dW, db, dx) for the mean cross-entropy.
    """
    # TODO 3: forward pass and loss
    # TODO 4: dlogits from ce_grad_logits, then the chain rule for W, b and x
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn.functional as F

torch.manual_seed(0)
N, D, V = 8, 5, 11

# 1. Jacobian of softmax
z = torch.randn(V, dtype=torch.float64)
J_ref = torch.autograd.functional.jacobian(lambda t: torch.softmax(t, -1), z)
with torch.no_grad():
    J = softmax_jacobian(z)
assert J.shape == (V, V)
assert torch.allclose(J, J_ref, atol=1e-10), "Jacobian does not match autograd"
assert torch.allclose(J.sum(0), torch.zeros(V, dtype=torch.float64), atol=1e-10), "columns of J sum to 0"

# 2. gradient of mean cross-entropy with respect to logits
logits = torch.randn(N, V, dtype=torch.float64, requires_grad=True)
targets = torch.randint(0, V, (N,))
F.cross_entropy(logits, targets).backward()
with torch.no_grad():
    g = ce_grad_logits(logits.detach().clone(), targets)
assert torch.allclose(g, logits.grad, atol=1e-10), "expected (p - y) / N"
assert torch.allclose(g.sum(-1), torch.zeros(N, dtype=torch.float64), atol=1e-10), "each row of p - y sums to 0"

# 3. one linear layer, all three gradients
x = torch.randn(N, D, dtype=torch.float64, requires_grad=True)
W = torch.randn(V, D, dtype=torch.float64, requires_grad=True)
b = torch.randn(V, dtype=torch.float64, requires_grad=True)
ref_loss = F.cross_entropy(x @ W.T + b, targets)
ref_loss.backward()
with torch.no_grad():
    loss, dW, db, dx = linear_ce_backward(x.detach(), W.detach(), b.detach(), targets)
assert torch.allclose(loss, ref_loss, atol=1e-10)
assert torch.allclose(dW, W.grad, atol=1e-10), "dW should be dlogits^T x"
assert torch.allclose(db, b.grad, atol=1e-10), "db should be the column sums of dlogits"
assert torch.allclose(dx, x.grad, atol=1e-10), "dx should be dlogits W"
print("m13_backward: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

雅可比矩阵是 `torch.diag(p) - torch.outer(p, p)`。

The Jacobian is `torch.diag(p) - torch.outer(p, p)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`ce_grad_logits` 先算 `p`，在正确类别的位置减 1，再除以 `N`。除以 `N` 是因为损失对 batch 取了平均。

In `ce_grad_logits` compute `p`, subtract 1 at the correct class, then divide by `N`. The division comes from averaging the loss over the batch.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

线性层的三个梯度可以靠形状推出来。`dW` 的形状是 `(V, D)`，只能是 `dlogits.T @ x`。`db` 的形状是 `(V,)`，是 `dlogits` 按行求和。`dx` 的形状是 `(N, D)`，是 `dlogits @ W`。

The three gradients of the linear layer follow from the shapes. `dW` is `(V, D)`, so it must be `dlogits.T @ x`. `db` is `(V,)`, the sum of `dlogits` over rows. `dx` is `(N, D)`, which is `dlogits @ W`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def softmax_jacobian(z):
    p = torch.softmax(z, dim=-1)
    return torch.diag(p) - torch.outer(p, p)          # p_i (delta_ij - p_j)


def ce_grad_logits(logits, targets):
    N = logits.shape[0]
    g = torch.softmax(logits, dim=-1)                 # p
    g[torch.arange(N), targets] -= 1.0                # p - y
    return g / N                                      # mean over the batch


def linear_ce_backward(x, W, b, targets):
    N = x.shape[0]
    logits = x @ W.T + b                              # (N, V)
    log_p = torch.log_softmax(logits, dim=-1)
    loss = -log_p[torch.arange(N), targets].mean()
    dlogits = ce_grad_logits(logits, targets)         # (N, V)
    dW = dlogits.T @ x                                # (V, D)
    db = dlogits.sum(dim=0)                           # (V,)
    dx = dlogits @ W                                  # (N, D)
    return loss, dW, db, dx
```

</details>

## 3. 推导 softmax 的梯度 / Derive the gradient of softmax

这是面试里出现很多的一道推导题。合上上面的推导，在纸上完成。
(a) 推导 $\partial p_i/\partial z_j$，分 $i=j$ 和 $i\neq j$ 两种情况。
(b) 推导交叉熵 $\mathcal L = -\sum_k y_k\log p_k$ 对 $z_j$ 的梯度。
(c) 若 softmax 带温度 $T$，梯度如何变化？
(d) 取 $z = (2,\ 1,\ 0.1)$，正确类别是第一个。算出 $p$、损失和梯度。

This derivation comes up a lot in interviews. Close the derivation above and work on paper.
(a) Derive $\partial p_i/\partial z_j$ for the cases $i=j$ and $i\neq j$.
(b) Derive the gradient of the cross-entropy $\mathcal L = -\sum_k y_k\log p_k$ with respect to $z_j$.
(c) How does the gradient change when softmax has temperature $T$?
(d) Let $z = (2,\ 1,\ 0.1)$ with the first class correct. Compute $p$, the loss and the gradient.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 用商的求导法则，记 $S = \sum_k e^{z_k}$，注意 $\partial S/\partial z_j = e^{z_j}$。

(a) Use the quotient rule with $S = \sum_k e^{z_k}$ and note $\partial S/\partial z_j = e^{z_j}$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(b) 更省事的做法是先写成 $\mathcal L = -z_c + \log\sum_k e^{z_k}$ 再求导。

(b) It is quicker to write $\mathcal L = -z_c + \log\sum_k e^{z_k}$ first and then differentiate.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) $e^{2}\approx 7.389$，$e^{1}\approx 2.718$，$e^{0.1}\approx 1.105$。

(d) $e^{2}\approx 7.389$, $e^{1}\approx 2.718$, $e^{0.1}\approx 1.105$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $i=j$ 时 $\dfrac{\partial p_i}{\partial z_i} = \dfrac{e^{z_i}S - e^{z_i}e^{z_i}}{S^2} = p_i(1-p_i)$。$i\neq j$ 时 $\dfrac{\partial p_i}{\partial z_j} = \dfrac{-e^{z_i}e^{z_j}}{S^2} = -p_ip_j$。合并为 $p_i(\delta_{ij}-p_j)$。

(b) 设正确类别为 $c$，$\mathcal L = -\log p_c = -z_c + \log\sum_k e^{z_k}$，于是 $$\frac{\partial\mathcal L}{\partial z_j} = -\delta_{jc} + \frac{e^{z_j}}{\sum_k e^{z_k}} = p_j - y_j$$

(c) 令 $u = z/T$，则 $\partial\mathcal L/\partial u_j = p_j - y_j$，再乘 $\partial u_j/\partial z_j = 1/T$，得到 $(p_j - y_j)/T$，其中 $p$ 是带温度的概率。

(d) 指数之和为 $7.389 + 2.718 + 1.105 = 11.212$，所以 $p \approx (0.659,\ 0.242,\ 0.099)$。损失为 $-\ln 0.659 \approx 0.417$。梯度为 $p - y \approx (-0.341,\ 0.242,\ 0.099)$，三个分量之和为 0。

---

(a) For $i=j$ we get $\dfrac{\partial p_i}{\partial z_i} = \dfrac{e^{z_i}S - e^{z_i}e^{z_i}}{S^2} = p_i(1-p_i)$. For $i\neq j$ we get $\dfrac{\partial p_i}{\partial z_j} = \dfrac{-e^{z_i}e^{z_j}}{S^2} = -p_ip_j$. Together, $p_i(\delta_{ij}-p_j)$.

(b) With correct class $c$, $\mathcal L = -\log p_c = -z_c + \log\sum_k e^{z_k}$, so $$\frac{\partial\mathcal L}{\partial z_j} = -\delta_{jc} + \frac{e^{z_j}}{\sum_k e^{z_k}} = p_j - y_j$$

(c) Let $u = z/T$. Then $\partial\mathcal L/\partial u_j = p_j - y_j$, and multiplying by $\partial u_j/\partial z_j = 1/T$ gives $(p_j - y_j)/T$, where $p$ is the tempered probability.

(d) The exponentials sum to $7.389 + 2.718 + 1.105 = 11.212$, so $p \approx (0.659,\ 0.242,\ 0.099)$. The loss is $-\ln 0.659 \approx 0.417$. The gradient is $p - y \approx (-0.341,\ 0.242,\ 0.099)$, and its components sum to 0.

</details>